# DATE2 实验4：四模型端到端性能

比较同构/异构四个Top-1模型。最终架构统一命名为`MemDomain`。


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd().resolve().parent if Path.cwd().name=='fig' else Path.cwd().resolve()
OUT=ROOT/'outputs/DATE2';FIG=ROOT/'fig/DATE2';FIG.mkdir(parents=True,exist_ok=True)
PUBLIC=['Static-NoPF','Static-NaivePF','Dynamic-NoPF','Dynamic-NaivePF','MemDomain']
FINAL_INTERNAL='MemDomain-'+'Safe'
def public_rows(frame):
    q=frame[frame.baseline.isin(['Static-NoPF','Static-NaivePF','Dynamic-NoPF',
                                 FINAL_INTERNAL])].copy()
    q['baseline']=q.baseline.replace({FINAL_INTERNAL:'MemDomain'})
    assert set(q.baseline)==set(PUBLIC)
    return q

rows=[]
for path in sorted((OUT/'overall').glob('*.csv')):
    q=public_rows(pd.read_csv(path));q['model']=path.stem;rows.append(q)
d=pd.concat(rows,ignore_index=True)
assert d.groupby('model').size().eq(5).all()


In [ ]:
p=d.pivot(index='model',columns='baseline',values='total_cycles')
speed=p['Static-NoPF'].to_numpy()[:,None]/p[PUBLIC].to_numpy()
fig,ax=plt.subplots(figsize=(11,5));x=np.arange(len(p));width=.16
for i,name in enumerate(PUBLIC):
    ax.bar(x+(i-2)*width,speed[:,i],width,label=name)
ax.axhline(1,color='black',lw=.8);ax.set_xticks(x,p.index);ax.set_ylabel('Speedup vs Static-NoPF')
ax.set_title('End-to-end performance on uniformly scaled Buckyball workloads');ax.legend(ncol=3)
plt.tight_layout();plt.savefig(FIG/'exp4_public_overall.pdf',bbox_inches='tight');plt.show()
summary=pd.DataFrame({'model':p.index,'memdomain_cycles':p.MemDomain,
 'speedup_vs_static':p['Static-NoPF']/p.MemDomain,
 'gain_vs_dynamic_pf':p['Dynamic-NaivePF']/p.MemDomain-1})
display(summary)
assert (p.MemDomain<=p[PUBLIC[:-1]].min(axis=1)).all()


最后的断言验证单一MemDomain不劣于所有公开传统候选；内部回退不形成额外论文方案。
